# Ch.3 — Feature Scaling, Importance & Multicollinearity

**Goal:** SmartVal AI has 8 features and a $55k MAE from Ch.2. Before adding polynomial complexity (Ch.4) or regularisation (Ch.5), we need to answer:
- Why must we scale features before comparing their importance?
- Which features carry the most signal toward house value?
- Which features measure the same thing (and therefore compete for the same weights)?
- What does each diagnostic method reveal that the others miss?

By the end of this notebook you will have a **three-view feature dashboard** and specific action items for Ch.4 and Ch.5.

In [ ]:
# TODO: Implement this cell


## 0 · Feature Scaling

Before any importance ranking is possible, all features must be on a **common scale**. Raw weights from an unstandardized model are not comparable — a weight of 200 for `Population` (range 3–35,682) looks enormous next to a weight of 0.4 for `MedInc` (range 0.5–15), but that gap reflects the input scale, not the feature's importance.

Think of this like comparing heights measured in millimeters vs kilometers — a 1.8m person becomes 1,800 mm but 0.0018 km. The number changes dramatically, but the person stays exactly the same height. Similarly, a feature's raw weight depends entirely on what units you happened to choose for measurement. StandardScaler fixes this by converting every feature to the same unit: **"one typical swing in that feature across the dataset."** After standardization, a coefficient of 0.8 on income and 0.1 on population means income has genuinely 8× the effect per standard deviation of change — a fair comparison.

**Standardization (Z-score):** $x_j^{\text{std}} = \dfrac{x_j - \mu_j}{\sigma_j}$

After standardization every feature has mean = 0, std = 1, so weight magnitudes are directly comparable.

> ⚠️ **Pipeline rule:** Always fit the scaler on **training data only**, then `transform` both train and test. Fitting on the full dataset leaks test statistics into training.

In [ ]:
# TODO: Implement this cell


## 0b · Variance Threshold — Dropping Near-Constant Features

A feature that barely changes gives the model nothing to latch onto — it's like trying to predict house prices using a column that says "2.00" for every district. Before proceeding with importance ranking, we check that all features have sufficient variance.

In [ ]:
# TODO: Implement this cell


## 0c · Understanding Positive Skew — When Features Need Log Transform

**Positive skew** means a distributional asymmetry with a long right tail — most values cluster near the low end, but a few extreme values stretch far to the right. This breaks StandardScaler because the few extreme values inflate the standard deviation (σ), compressing 95% of the data into a narrow band near zero while placing outlier districts at +8σ or +12σ.

**Example:** `Population` ranges from 3 to 35,682. Let's visualize the problem and the log-transform solution.

In [ ]:
# TODO: Implement this cell


## 0d · Weight-Magnitude Comparison — The 28,000× Gap is a Scale Artifact

Let's demonstrate why raw weights can't be compared. We'll fit two models — one on raw features, one on scaled — and show that the weight gap is almost entirely due to input scale, not importance.

In [ ]:
# TODO: Implement this cell


## 1 · Fit the Baseline Model

Data is already loaded and scaled from Section 0. We just fit the Ch.2 LinearRegression on the standardized splits.

In [ ]:
# TODO: Implement this cell


## 1b · Three-Lens Framework — How to Interpret Feature Rankings

We'll measure feature importance using **three methods**. Each answers a different question, and where their rankings diverge tells the richest diagnostic story:

| Univariate R² | Methods 2+3 (Joint) | Interpretation | Example |
|---|---|---|
| **High** | **High** | ✅ Strong, independent, irreplaceable | MedInc — dominates alone *and* in joint model |
| **High** | **Low** | ⚠️ Signal shared with correlated features | AveRooms — standalone power absorbed by AveBedrms |
| **Low** | **High** | 🔵 Jointly irreplaceable — only works in combination | Lat/Lon — useless alone, critical together for geography |
| **Low** | **Low** | ❌ Genuinely uninformative | Population — contributes ~$0 regardless of method |

**Armed with this framework, let's collect all three views:**

## 1c · Filter Methods — Pearson vs Mutual Information

Before fitting any model we can get a directional signal from **filter statistics** — computed directly from the data.

**Pearson ρ** captures linear associations. **Mutual Information** captures *any* statistical dependence: curves, U-shapes, thresholds, clusters.

$$I(X;Y) = \iint p(x,y)\,\log\frac{p(x,y)}{p(x)\,p(y)}\,dx\,dy$$

$p(x,y)$ is the joint density — a 2-D map of where the scatter concentrates. $p(x) \cdot p(y)$ is the independence baseline. The log-ratio measures, at every point, how far the actual density deviates from independence; MI sums those deviations over the whole plane.

**When to use which:**
- **Pearson** — linear or near-linear; score directly comparable to R²
- **MI** — any shape; essential before non-linear models and as a sanity check for linear ones

> ⚙️ **How `sklearn` estimates MI for continuous variables.** `mutual_info_regression` uses the **Kraskov k-NN estimator**: for each sample it measures the distance to its *k*-th nearest neighbour in joint (x, y) space versus in the separate marginal spaces — no binning, no histogram choice. Key settings: `n_neighbors` controls bias-variance (default 3: lower bias but noisier; larger k smooths at the cost of resolution). Always set `random_state` — the estimator adds a tiny perturbation to break ties. MI scores are **not normalised** — use relative magnitude only, never compare across datasets.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 2 · Method 1 — Univariate R²

**Question:** If I used only this feature, how much target variance would it explain?

**Shortcut:** For linear regression, univariate R² = ρ(xⱼ, y)². Read it from the correlation matrix — no need to fit 8 models.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 3 · Method 2 — Standardised Weights (Partial Contribution)

**Question:** Given all other features in the model, how much does each feature contribute?

This is the *partial* effect — what each feature adds above and beyond everything else.

In [ ]:
# TODO: Implement this cell


## 4 · The Surprise — Why the Rankings Diverge

Let's directly visualise the ranking gap between the two methods.

In [ ]:
# TODO: Implement this cell


## 5 · Feature Correlation Heatmap

Visualising *feature × feature* correlations reveals the collinear pairs **before** computing VIF.

In [ ]:
# TODO: Implement this cell


## 6 · VIF — Quantifying Multicollinearity

VIF measures how much each feature's weight is inflated by its correlation with the **other features**.

$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$

where $R^2_j$ is the R² from regressing feature $j$ on **all other features** (not the target).

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 7 · Method 3 — Permutation Importance

**Question:** If I *scramble* this feature on the test set (destroying its signal), how much does MAE rise?

This is the **most reliable** and model-agnostic method. Crucially, the model is never retrained — you're measuring how badly the model's existing weights are handicapped when a feature's signal is destroyed. This makes it a pure test of the model's *reliance* on each feature, not just correlation or fitted weights.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 7b · Permutation Shuffle Loop — Visualizing How It Works

Let's demonstrate the shuffle loop on a small sample to show exactly what permutation importance measures. We'll take 10 test samples, shuffle one feature (MedInc), and show that predictions change even though the model weights stay frozen.

In [ ]:
# TODO: Implement this cell


## 8 · Three-View Dashboard

Side-by-side comparison of all three methods. Features where rankings diverge tell the richest story.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 9 · Multicollinearity Deep Dive — AveRooms vs AveBedrms

Visualise *why* the two room features cause weight instability.

In [ ]:
# TODO: Implement this cell


## 9b · Joint Feature Importance — Cooperation vs Competition

VIF catches the *competition* case: two features measure the same thing and their weights blow up. The opposite exists too: two features can be **individually weak but jointly irreplaceable** — the *cooperation* case.

**Diagnostic — joint permutation importance**: shuffle both features simultaneously and compare the drop to the sum of their individual drops.

$$\Delta_{\text{interact}}(j,k) = \pi_{jk} - \pi_j - \pi_k$$

- $\Delta > 0$ → the model is using their *interaction*; the features cooperate (Lat/Long case)
- $\Delta \approx 0$ → additive independence
- $\Delta < 0$ → the features are substitutes; one is largely redundant (AveRooms/AveBedrms case)

In [ ]:
# TODO: Implement this cell


## 10 · Action Items for Ch.4 & Ch.5

Based on the three-view dashboard, what do we know about pursuing the $40k MAE target?

In [ ]:
# TODO: Implement this cell


## Summary

| Method | Top feature | Bottom feature | Best use |
|---|---|---|---|
| Univariate R² | MedInc (0.47) | Population (0.001) | First-pass scan; quick ranking |
| Std \|weight\| | Latitude (0.89) | Population (0.01) | Final model inspection; requires standardisation |
| Permutation | MedInc (+$18k) | Population (+$0.1k) | Most reliable; model-agnostic; use on test set |

**Key takeaways:**
1. **MedInc** is the strongest individual predictor regardless of method.
2. **Latitude and Longitude** are jointly irreplaceable — low alone, high together.
3. **AveRooms / AveBedrms** are collinear (VIF ≈ 7) — their weight split is arbitrary.
4. **Population** contributes virtually nothing and is a candidate for removal.
5. Always inspect all three views; a feature cheap in one can be critical in another.

**Next:** Ch.4 — Polynomial Features: add `MedInc²`, `MedInc × Latitude`, `MedInc × Longitude` to push MAE toward $48k.